# 冻结规则诊断 · 可复算证据
**TL;DR：** 10例正样本均进入过S1，6例全窗有S2。4例负样本事件有S2，只有1例事件内新确认。G5仍FAIL；25例仅作发展/诊断集。

## Context / Methods
仅读原始冻结casebook、快照/规则/状态；不重评规则、不调用provider、不写Production。频率按案例等权，NULL保留分母。案例存在行业/时间相关性，不是独立试验。确认前窗口排除确认日；确认后仅作事后解释。

In [1]:
from pathlib import Path
import sys,json
ROOT=Path.cwd()
sys.path.insert(0,str(ROOT/'scripts/mainline'))
from verify_rule_diagnostic import verify
D=ROOT/'reports/milestone-d-rule-diagnostic'
checks=verify()
print('Independent raw-archive checks:',checks['status'])
print(json.dumps(checks['headline_recomputed'],ensure_ascii=False,indent=2))

Independent raw-archive checks: PASS
{
  "positive_event_days": 244,
  "positive_full_window_S1": 10,
  "positive_full_window_S2": 6,
  "positive_event_window_S2": 5,
  "case_days": 1631,
  "transitions": 341,
  "freeze_days": 63,
  "negative_event_days": 88,
  "negative_any_event_S2_cases": 4,
  "negative_false_S2_days": 11,
  "ambiguous_event_days": 118
}


## Data sample
前10个案例沿用固定标签和窗口，不改expected_behavior。原子逐日CSV另存actual、threshold、signed margin、NULL与描述性边缘标识。

In [2]:
book=json.loads((ROOT/'reports/milestone-d-baseline-v1/casebook.json').read_text())
for c in book['cases'][:10]:
    print(c['case_id'],c['case_type'],c['sector'],c['start_date'],c['end_date'])
print('Frozen checksum:',book['checksum'])
print('Frozen business files verified:',checks['frozen_files_verified'])

P01 positive 电子 2024-10-18 2024-11-12
P02 positive 电力设备 2024-10-18 2024-11-12
P03 positive 银行 2024-11-18 2024-12-31
P04 positive 商贸零售 2024-11-25 2024-12-16
P05 positive 社会服务 2024-11-25 2024-12-16
P06 positive 计算机 2025-02-05 2025-03-14
P07 positive 机械设备 2025-02-05 2025-03-18
P08 positive 汽车 2025-02-05 2025-03-18
P09 positive 有色金属 2025-05-06 2025-06-13
P10 positive 国防军工 2025-05-06 2025-06-13
Frozen checksum: 249799d4c82ee067c4450e58d3409c661e2afb0dc7ff0339ae277f0f6d69c47d
Frozen business files verified: 7


## Results
B唯一阻塞25日/7例，有S1合法升级资格的仅8日。D.newhigh/E3在450个事件案例日完全重复。候选失败日票数不能当作改阈值的净收益。

In [3]:
print('B sole cases:',checks['B_sole_cases'])
print('Blockers / eligible:',checks['sole_blocker_days'])
print('New event negative confirmations:',checks['new_negative_confirmations'])
print('D.newhigh/E3 identical rows:',checks['D_newhigh_E3_identical_event_rows'])
print('Candidate failure votes:',checks['candidate_failure_votes'])
print('Previous votes:',checks['previous_candidate_votes'])
for r in json.loads((D/'persistence_summary.json').read_text()):
    if r['window'] in [5,10,20]:
        print(r['cohort'],r['window'],'n=',r['cases'],'daily win=',round(r['daily_relative_win_rate'],3),'C pass=',round(r['C_pass_frequency'],3))

B sole cases: ['P03', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10']
Blockers / eligible: {'C': 7, 'C_eligible': 2, 'B': 25, 'B_eligible': 8, 'A': 2, 'A_eligible': 1, 'enhancers': 1}
New event negative confirmations: ['N10']
D.newhigh/E3 identical rows: 450
Candidate failure votes: {0: 10, 2: 80, 1: 48}
Previous votes: {1: 1, 4: 37, 3: 92, 0: 1, 5: 4}
正样本事件期S2 5 n= 5 daily win= 0.76 C pass= 0.64
正样本事件期S2 10 n= 5 daily win= 0.64 C pass= 0.52
正样本事件期S2 20 n= 5 daily win= 0.52 C pass= 0.4
负样本事件期S2 5 n= 4 daily win= 0.85 C pass= 0.55
负样本事件期S2 10 n= 4 daily win= 0.65 C pass= 0.375
负样本事件期S2 20 n= 4 daily win= 0.55 C pass= 0.325


## Takeaways / Next steps
高胜率否决、成交持续性不足、证据重复与候选反复值得有限研究，不能统一放宽门槛。确认前比较仅5正/4负，P05/N07共享同一次确认。完整warmup缺失使状态结论WARMUP_SENSITIVE；无独立退潮参考日期，不能判S3/S4早晚。

Top5候选和新10正/10负/5模糊Holdout设计见报告，本轮不选择新案例、不执行校准。production_writes=0；effective_pit，knowledge_time_unverified=true。